# 02 · Generate fakes from ABO (pilot first, then SDXL-Turbo, SD 1.5, Kandinsky 2.2)

**Accelerator:** GPU **T4 x2** · **Attach:** `abo-prepared` · **Output dataset name:** `abo-fakes-<run with dashes>`

Set `RUN` and commit once per run. **Do the `pilot` run first** and take it through notebooks 04 → 05 → 06
before spending the rest of the budget (runbook, "Pilot").

| RUN | generator | mode | strength | images (train/val/test) | note |
|---|---|---|---|---|---|
| `pilot` | sdxl-turbo | t2i **and** i2i 0.5 | – | 300/50/100 each | ~15 min; the vertical slice |
| `turbo_t2i` | sdxl-turbo | t2i | – | 2100/300/600 | |
| `turbo_i2i` | sdxl-turbo | i2i | 0.5 | 2100/300/600 | 2 effective denoising steps |
| `sd15_i2i`  | sd15 | i2i | 0.5 | 1400/200/400 | gated: HF_TOKEN secret |
| `kand_t2i`  | kandinsky22 | t2i | – | 700/100/200 | third family, unseen for `main` |
| `kand_i2i`  | kandinsky22 | i2i | 0.5 | 700/100/200 | |

Timings are estimates until the pilot measures them: benchmark seconds/image from the pilot log and scale.

**Why fakes from ABO itself:** fakes that differ in *style* (clutter, framing, resolution) would let the probe learn
"catalogue photo vs not" (Risk 1). Same-domain generation reduces that confound; it does not remove it, which is why
notebook 04 runs a nuisance-only audit and a style-mismatch control.

**Geometry:** the i2i source is letterboxed (never stretched) and every fake is cropped back to its source listing's
aspect ratio, so shape carries no label. Every 10th fake keeps a lossless PNG twin for the storage diagnostic.

`--gpus auto` spawns one worker per visible GPU (chunks split by index modulo). Re-running the same RUN resumes.
Each run writes to its own folder (`<generator>/<mode>/n<sizes>_seed<seed>/`), so the pilot and the full run of
the same generator never share chunk files when their datasets are merged in notebook 04.

## 0. Setup (same cell in every notebook)

**Before running:** attach the code as a Kaggle Dataset named `listing-verifier-code`
(upload the repo folder without `data/`, `outputs/`, `web/node_modules/`), **or** set `REPO_URL`
to your GitHub repo. Turn on **Internet** (pip, ABO and HF downloads). For the GPU notebooks pick
**Settings → Accelerator → GPU T4 x2**. Store your Hugging Face token as the Kaggle secret `HF_TOKEN`.

In [ ]:
import os, shutil, subprocess, sys, pathlib, json
REPO_URL = "https://github.com/YOUR_USER/listing-verifier.git"   # <- only used if the code dataset is not attached
WORK = pathlib.Path("/kaggle/working"); REPO = WORK / "repo"; DATA = WORK / "data"; OUT = WORK / "outputs"
code_ds = pathlib.Path("/kaggle/input/listing-verifier-code")
if not REPO.exists():
    if code_ds.exists():
        shutil.copytree(code_ds, REPO)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
os.chdir(REPO)
os.environ["LV_DATA_ROOT"] = str(DATA); os.environ["LV_OUTPUT_ROOT"] = str(OUT)
os.environ["HF_HOME"] = str(WORK / "hf_cache")           # model weights land here (removed before publishing)
DATA.mkdir(exist_ok=True); OUT.mkdir(exist_ok=True)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    print("no HF_TOKEN secret:", type(e).__name__, "(fine unless a gated model is needed)")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "check"], check=False)

LOGS = OUT / "logs"; LOGS.mkdir(parents=True, exist_ok=True)

def run(*args):
    """Run a pipeline script; stream its output to the cell AND to OUT/logs/<script>_<time>.log,
    append a completion record to OUT/logs/completed.jsonl, and stop the notebook on a non-zero exit."""
    import time, datetime
    cmd = [sys.executable, *[str(a) for a in args]]
    name = pathlib.Path(str(args[0])).stem
    stamp = datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
    log_path = LOGS / f"{name}_{stamp}.log"
    print(">>>", " ".join(cmd[1:]), "->", log_path.name, flush=True)
    t0 = time.time()
    with open(log_path, "w") as log:
        log.write("$ " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True); log.write(line)
        rc = proc.wait()
    record = {"script": name, "args": cmd[2:], "returncode": rc, "seconds": round(time.time() - t0, 1),
              "log": str(log_path), "finished_utc": datetime.datetime.utcnow().isoformat() + "Z"}
    with open(LOGS / "completed.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

def attach(slug, sub, dest, must_contain=None):
    """Copy a sub-folder of an attached Kaggle dataset into DATA/OUT (copy, not symlink: scripts write next to inputs).
    Refuses silently stale copies: if `must_contain` is missing from an existing dest, the copy is redone."""
    base = pathlib.Path("/kaggle/input") / slug
    src = base / sub if (base / sub).exists() else base
    dst = pathlib.Path(dest)
    if dst.exists() and (must_contain is None or (dst / must_contain).exists()):
        print("exists:", dst); return
    if dst.exists():
        shutil.rmtree(dst)
    if not src.exists():
        raise FileNotFoundError(f"dataset {slug} is not attached (expected {src})")
    print("copy", src, "->", dst); shutil.copytree(src, dst)

run("scripts/env_report.py", "--out", OUT / "results" / "env_report.json")
import torch; print(f"visible GPUs: {torch.cuda.device_count()}")


In [ ]:
RUN = "pilot"   # pilot | turbo_t2i | turbo_i2i | sd15_i2i | kand_t2i | kand_i2i
RUNS = {"turbo_t2i": [["--generator", "sdxl-turbo", "--mode", "t2i", "--per-split", "train=2100,val=300,test=600"]],
        "turbo_i2i": [["--generator", "sdxl-turbo", "--mode", "i2i", "--strength", "0.5", "--per-split", "train=2100,val=300,test=600"]],
        "sd15_i2i":  [["--generator", "sd15", "--mode", "i2i", "--strength", "0.5", "--per-split", "train=1400,val=200,test=400"]],
        "kand_t2i":  [["--generator", "kandinsky22", "--mode", "t2i", "--per-split", "train=700,val=100,test=200"]],
        "kand_i2i":  [["--generator", "kandinsky22", "--mode", "i2i", "--strength", "0.5", "--per-split", "train=700,val=100,test=200"]],
        "pilot":     [["--generator", "sdxl-turbo", "--mode", "t2i", "--per-split", "train=300,val=50,test=100"],
                      ["--generator", "sdxl-turbo", "--mode", "i2i", "--strength", "0.5", "--per-split", "train=300,val=50,test=100"]]}
attach("abo-prepared", "data/processed", DATA / "processed", must_contain="abo_clean.csv")
import time
for job in RUNS[RUN]:
    t0 = time.time(); run("scripts/03_generate_fakes.py", *job, "--chunk-size", 500, "--gpus", "auto")
    print(f"[timing] {job[1]} {job[3]}: {time.time() - t0:.0f} s wall")

### Look at real vs fake side by side, and check failed images (black, text overlays, distortions). Log any exclusion rule BEFORE looking at detector scores.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
csvs = sorted(glob.glob(str(DATA / "fakes" / "**" / "chunk_*.csv"), recursive=True)); allf = pd.concat(map(pd.read_csv, csvs))
fk = allf.sample(min(6, len(allf)), random_state=0)
abo = pd.read_csv(DATA / "processed" / "abo_clean.csv").set_index("item_id")
fig, axes = plt.subplots(2, len(fk), figsize=(3 * len(fk), 6))
for i, (_, r) in enumerate(fk.iterrows()):
    axes[0, i].imshow(Image.open(DATA / "processed" / "images" / abo.loc[r["item_id"], "image_rel_path"])); axes[0, i].set_title("real", fontsize=9)
    axes[1, i].imshow(Image.open(r["path"])); axes[1, i].set_title(f"{r['generator']} {r['mode']} steps={r.get('effective_steps', '?')}", fontsize=8)
    axes[0, i].axis("off"); axes[1, i].axis("off")
plt.tight_layout(); print(len(allf), "fakes so far;", "png twins:", int(allf.get("png_twin", pd.Series(dtype=bool)).sum()))
print(allf.groupby(["generator", "mode"]).size())

## Persisting outputs
When this notebook finishes under **Save Version → Save & Run All (Commit)**, everything in `/kaggle/working`
except the model cache is kept as the notebook output. Then: *Output tab → "New Dataset"* and name it as stated
at the top of this notebook. The next notebook attaches that dataset. The cell below checks the artifacts this
notebook promised actually exist, then removes the model cache so the output stays small.

In [ ]:
required = ["data/fakes"]
run("scripts/verify_outputs.py", *[WORK / r for r in required])   # exits non-zero on a missing/empty artifact
shutil.rmtree(WORK / "hf_cache", ignore_errors=True)
shutil.rmtree(REPO / "web" / "node_modules", ignore_errors=True)
print(subprocess.run(["du", "-sh", str(DATA), str(OUT)], capture_output=True, text=True).stdout)